# ch09 — Industrial practice: the BYOD workflow

Evaluate your own CSV end to end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
# Create an example CSV (use your own file path in practice)
import pandas as pd
rng = np.random.default_rng(0)
n = 3000
values = np.sin(2*np.pi*np.arange(n)/100) + rng.normal(scale=0.1, size=n)
labels = np.zeros(n, dtype=int)
values[2400:2450] += 2.5
labels[2400:2450] = 1
pd.DataFrame({"timestamp": np.arange(n), "value": values, "label": labels}).to_csv(
    "/tmp/my_sensor.csv", index=False)

In [ ]:
from tsad_forge.cli import main
# With a label column the full metric suite is computed;
# without one you get scores + threshold decisions only.
main(["run", "--model", "sub_pca", "--data", "/tmp/my_sensor.csv",
      "--results-dir", "/tmp/byod-results"])

In [ ]:
# Regime vs fault: a regime change (drift) is not a fault — compare drift-aware thresholds
from tsad_forge.evaluation.thresholding import spot_threshold, dspot_threshold
drifting = np.arange(3000)/300 + rng.normal(scale=0.5, size=3000)
print("SPOT :", round(spot_threshold(drifting, q=1e-3), 2), "(mistakes the drift for anomaly)")
print("DSPOT:", round(dspot_threshold(drifting, q=1e-3, depth=100), 2), "(tracks the final drift level)")